# Reinforced Graph of Thoughts - PPO - Task Count Keywords

In [1]:
import pandas as pd
from pure_graph_of_thoughts.api.language_model import Example

from reinforced_graph_of_thoughts.tasks.count_keywords import create_count_keywords_task, create_op_count

countries_url = 'https://raw.githubusercontent.com/spcl/graph-of-thoughts/refs/heads/main/examples/keyword_counting/countries.csv'
countries = pd.read_csv(countries_url)
text_keyword_map = (
    countries.dropna(subset=['Countries'])
    .assign(Countries=lambda df: df['Countries'].apply(
        lambda x: x[1:-1].split(', ') if isinstance(x, str) and len(x) > 1 else []))
    .set_index('Text')['Countries']
    .to_dict()
)
keywords = list(set([
    item for value in text_keyword_map.values() for item in value
]))
all_texts = list(text_keyword_map.keys())

op_count_keywords = create_op_count(
    instruction='Count the occurrence of countries in the given text.',
    examples=[
        Example(
            input={
                'text': 'France and Italy are known for their rich cultural heritage and exquisite cuisine, while Japan offers a blend of ancient tradition and cutting-edge technology. Meanwhile, Italy’s scenic countryside, Brazil’s vibrant festivals and Australia’s stunning landscapes attract travelers from around the world.'
            },
            output={
                'count': {
                    'France': 1,
                    'Italy': 2,
                    'Japan': 1,
                    'Brazil': 1,
                    'Australia': 1,
                    'Switzerland': 1
                }
            }
        )
    ],
    keywords=keywords
)
count_keywords_task = create_count_keywords_task(keywords, op_count_keywords)

In [ ]:
from reinforced_graph_of_thoughts.experiment.experiment_task_type import ExperimentTaskType
from reinforced_graph_of_thoughts.agent.train_agent import train_agent
from reinforced_graph_of_thoughts.experiment import create_generate_init_state_count_keywords

TASK_NAME = 'count_keywords'

TRAIN_COMPLEXITIES = [c for c in range(10, 50 + 1, 10)]
EVAL_COMPLEXITIES = [c for c in range(10, 100 + 1, 10)]

SEED = 0

ARTIFACTS_BASE_DIR = '../artifacts'
RESULTS_DIR = f'{ARTIFACTS_BASE_DIR}/results/agent_evaluations'

model_name = train_agent(
    seed=SEED,
    artifacts_base_dir=ARTIFACTS_BASE_DIR,
    task_name=TASK_NAME,
    task=count_keywords_task,
    train_complexities=TRAIN_COMPLEXITIES,
    eval_complexities=EVAL_COMPLEXITIES,
    generate_init_state=create_generate_init_state_count_keywords(all_texts),
    task_type=ExperimentTaskType.COUNT_KEYWORDS,
    extra_args={
        'keywords': keywords,
        'op_count': op_count_keywords
    }
)


In [ ]:
from reinforced_graph_of_thoughts.experiment.visualization_utils import visualize_solved_rate, visualize_avg_n_operations
from reinforced_graph_of_thoughts.experiment.evaluation_summary_utils import load_evaluation_summary

evaluation_summary = load_evaluation_summary(RESULTS_DIR, model_name)
visualize_solved_rate(model_name, evaluation_summary.solved_rate_per_complexity)
visualize_avg_n_operations(model_name, evaluation_summary.avg_n_operations_per_complexity)